WEB_SCRAPING CR DATA FROM portal.3gpp.org/ChangeRequests for analysis

In [ ]:
import time
import re
from io import StringIO
from urllib.parse import urlencode

import pandas as pd
import requests

In [30]:
import re
import time
from io import StringIO
from urllib.parse import urlencode

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://portal.3gpp.org/ChangeRequests.aspx"

PARAMS = {
    "q": "1",
    "specnumber": "",
    # one or more release ids, comma separated. "193" = Rel-18 only.
    "release": "193",
    "wgstatus": "",
    "tsgstatus": "",
    "meeting": "",
    "workitem": "",
}

HEADERS = {"User-Agent": "Mozilla/5.0"}

COLUMN_HEADERS = [
    "Spec #",
    "CR #",
    "Revision #",
    "CR Cat",
    "Impacted Version",
    "Target Release",
    "Title",
    "WG TDoc #",
    "CR status at WG",
    "WG meeting ref",
    "WG Source information",
    "TSG TDoc #",
    "CR status at TSG",
    "TSG meeting ref",
    "TSG Source information",
    "New Version",
    "Work Items",
    "Remarks",
]

KEEP_COLUMNS = [
    "CR Cat",
    "Target Release",
    "Title",
    "WG TDoc #",
    "CR status at WG",
    "WG meeting ref",
    "WG Source information",
    "TSG TDoc #",
    "CR status at TSG",
    "TSG meeting ref",
    "TSG Source information",
    "New Version",
    "Work Items",
]

N_COLS = len(COLUMN_HEADERS)


# --------------------------------------------------------------------------
# networking
# --------------------------------------------------------------------------
def get_page(session, page_index):
    params = PARAMS.copy()
    params["pageindex"] = page_index
    url = f"{BASE_URL}?{urlencode(params)}"
    response = session.get(url, headers=HEADERS, timeout=60)
    response.raise_for_status()
    return response.text, url


def extract_total_pages(html):
    """Parse 'NNNN items in MM pages'."""
    text = BeautifulSoup(html, "html.parser").get_text(" ", strip=True)
    match = re.search(r"(\d+)\s+items\s+in\s+(\d+)\s+pages", text)
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None


# --------------------------------------------------------------------------
# parsing  (the important part)
# --------------------------------------------------------------------------
def _cell_text(cell):
    """Text of a single cell, with <br> turned into spaces, whitespace squashed."""
    text = cell.get_text(" ", strip=True)
    return re.sub(r"\s+", " ", text).strip()


def _row_cells(tr):
    """
    All direct cells of a row, IN ORDER, WITHOUT dropping empties.
    recursive=False so we don't pick up cells from nested tables.
    """
    cells = tr.find_all(["td", "th"], recursive=False)
    if not cells:  # some grids wrap cells one level deeper
        cells = tr.find_all(["td", "th"])
    return [_cell_text(c) for c in cells]


def _looks_like_header(cells):
    joined = " ".join(cells)
    return "Spec #" in joined and "CR #" in joined and "Target Release" in joined


_BAD_ROW_MARKERS = (
    "PageSizeComboBox",
    "Items per page",
    "Search form",
    "Data pager",
    "function ",
)

# control/utility column labels that the RadGrid adds and that we don't want
_CONTROL_NAMES = {"", "select", "data pager", "edit", "expand", "collapse"}


def _is_junk_row(cells):
    joined = " ".join(cells)
    if not joined.strip():
        return True
    if re.search(r"\d+\s+items\s+in\s+\d+\s+pages", joined):
        return True
    # a lone control cell on its own row (e.g. ['select'])
    if len(cells) <= 2 and all(c.strip().lower() in _CONTROL_NAMES for c in cells):
        return True
    return any(marker in joined for marker in _BAD_ROW_MARKERS)


def _align_to_header(cells, n_head):
    """
    Make a data row exactly n_head wide WITHOUT deleting interior empties.

    The RadGrid prepends control columns (pager / select) to every DATA row,
    so a data row is wider than the header row. Those control columns are on
    the LEFT, so we keep the RIGHTMOST n_head cells. Short rows are right-padded.
    """
    if len(cells) > n_head:
        return cells[len(cells) - n_head:]   # drop leading control columns
    if len(cells) < n_head:
        return cells + [""] * (n_head - len(cells))
    return cells


def extract_cr_table(html, debug=False):
    """
    Find the real CR grid and return a DataFrame with the 18 CR columns aligned.

    We map to the fixed COLUMN_HEADERS schema (which matches the grid's columns).
    Each data row is RIGHT-aligned to those 18 columns: the RadGrid prepends
    control columns (pager / select) to every data row, so the real data sits in
    the rightmost 18 cells. Empty cells are preserved so nothing shifts, and we
    never truncate real data off the right.
    """
    soup = BeautifulSoup(html, "html.parser")

    for table in soup.find_all("table"):
        rows = [_row_cells(tr) for tr in table.find_all("tr")]
        rows = [r for r in rows if r]

        header_index = next(
            (i for i, r in enumerate(rows) if _looks_like_header(r)), None
        )
        if header_index is None:
            continue

        if debug:
            print(f"[debug] header row ({len(rows[header_index])} cells): "
                  f"{rows[header_index]}")

        data_rows = []
        for raw in rows[header_index + 1:]:
            if _looks_like_header(raw) or _is_junk_row(raw):
                continue
            if debug and len(data_rows) < 3:
                print(f"[debug] raw data row ({len(raw)} cells): {raw}")
            data_rows.append(_align_to_header(raw, N_COLS))

        if data_rows:
            return pd.DataFrame(data_rows, columns=COLUMN_HEADERS)

    return None


def clean_text_columns(df):
    df = df.copy()
    for col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .replace({"nan": "", "None": ""})
        )
    return df


# --------------------------------------------------------------------------
# driver
# --------------------------------------------------------------------------
def download_all_crs(max_pages=None, sleep_seconds=0.5, debug=False):
    all_pages = []
    start = time.perf_counter()
    total_start_time = time.perf_counter()

    with requests.Session() as session:
        first_html, _ = get_page(session, page_index=0)
        total_items, total_pages = extract_total_pages(first_html)
        if total_pages is None:
            print("Could not detect total pages; using max_pages as fallback.")
            total_pages = 10
        else:
            print(f"Detected {total_items} items across {total_pages} pages.")

        if max_pages is not None:
            total_pages = min(total_pages, max_pages)

        for page_index in range(total_pages):
            page_start_time = time.perf_counter()
            print(f"Downloading page {page_index + 1}/{total_pages}")
            html = first_html if page_index == 0 else get_page(session, page_index)[0]

            table = extract_cr_table(html, debug=debug and page_index == 0)
            if table is None or table.empty:
                print(f"No CR table on page {page_index}; stopping.")
                break

            table = clean_text_columns(table)
            table["source_pageindex"] = page_index
            all_pages.append(table)
            time.sleep(sleep_seconds)
            page_end_time = time.perf_counter()
            page_seconds = page_end_time - page_start_time

            elapsed_seconds = page_end_time - total_start_time
            avg_seconds_per_page = elapsed_seconds / (page_index + 1)

            remaining_pages = total_pages - (page_index + 1)
            estimated_remaining_seconds = remaining_pages * avg_seconds_per_page

            print(
                f"Page {page_index + 1} done in {page_seconds:.2f} sec | "
                f"Elapsed: {elapsed_seconds / 60:.2f} min | "
                f"ETA: {estimated_remaining_seconds / 60:.2f} min"
            )
            total_end_time = time.perf_counter()
    total_seconds = total_end_time - total_start_time

    print("\nDownload finished.")
    print(f"Total time: {total_seconds:.2f} sec")
    print(f"Total time: {total_seconds / 60:.2f} min")
    if not all_pages:
        return pd.DataFrame()
    return pd.concat(all_pages, ignore_index=True)


if __name__ == "__main__":
    # TEST: first page only, with debug so you can see the real header + raw
    # row widths. Change max_pages to None to scrape everything.
    df = download_all_crs(max_pages=None, sleep_seconds=0.5, debug=False)
    print(f"\nRows downloaded: {len(df)}")
    print("Columns parsed:", list(df.columns))

    # keep only the columns you want, by name, ignoring any that are absent
    keep = [c for c in KEEP_COLUMNS if c in df.columns]
    df_final = df[keep].copy()
    df_final.to_csv("3gpp_relcrs.csv", index=False, encoding="utf-8-sig")
    print("Saved: 3gpp_relcrs.csv")
    with pd.option_context("display.max_columns", None, "display.width", 220):
        print(df_final.head(10))

Detected 521974 items across 2610 pages.
Page 1 done in 0.80 sec | Elapsed: 0.05 min | ETA: 125.62 min
Page 2 done in 2.16 sec | Elapsed: 0.08 min | ETA: 109.81 min
Page 3 done in 2.60 sec | Elapsed: 0.13 min | ETA: 110.81 min
Page 4 done in 2.39 sec | Elapsed: 0.17 min | ETA: 109.08 min
Page 5 done in 2.37 sec | Elapsed: 0.21 min | ETA: 107.83 min
Page 6 done in 2.37 sec | Elapsed: 0.25 min | ETA: 106.94 min
Page 7 done in 2.48 sec | Elapsed: 0.29 min | ETA: 106.98 min
Page 8 done in 2.50 sec | Elapsed: 0.33 min | ETA: 107.10 min
Page 9 done in 2.36 sec | Elapsed: 0.37 min | ETA: 106.54 min
Page 10 done in 2.44 sec | Elapsed: 0.41 min | ETA: 106.43 min
Page 11 done in 2.72 sec | Elapsed: 0.45 min | ETA: 107.44 min
Page 12 done in 2.93 sec | Elapsed: 0.50 min | ETA: 109.03 min
Page 13 done in 3.21 sec | Elapsed: 0.56 min | ETA: 111.27 min
Page 14 done in 2.64 sec | Elapsed: 0.60 min | ETA: 111.43 min
Page 15 done in 2.61 sec | Elapsed: 0.64 min | ETA: 111.50 min
Page 16 done in 2.68 se

In [29]:
df_final

,CR Cat,Target Release,Title,WG TDoc #,CR status at WG,WG meeting ref,WG Source information,TSG TDoc #,CR status at TSG,TSG meeting ref,TSG Source information,New Version,Work Items
0,A,Rel-9,Alert MME Request / UE-Activity-Indication pro...,C4-100860,postponed,CT4#48,Alcatel-Lucent,,,,C4,,SAES
1,A,Rel-9,Alert MME Request / UE-Activity-Indication pro...,C4-100399,revised,CT4#48,Alcatel-Lucent,,,,C4,,SAES
2,F,Rel-8,Zh and ZhÆ intradomain reference points,S3-080682,revised,SA3#52,Ericsson,,,,,,
3,F,Rel-8,Clarification on Annex A.9: Measurement perfor...,R4-090968,,,NEC,RP-090269,reissued,RAN#43,R4,,LTE-RF
4,F,Rel-7,Add missing EF in ISIM file structure,C6-060282,agreed,CT6#39,,CP-060243,approved,CT#32,C6,,TEI7
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1796,C,Rel-7,Clarification of VCC Triggers,S1-050936,agreed,SA1#30,,SP-050518,approved,SA#29,S1,,VCC
1797,C,Rel-7,Clarification of VCC Triggers,S1-050900,revised,SA1#29,,,,,S1,,VCC
1798,C,Rel-7,VCC call handling when two subscriptions exist,S1-050898,agreed,SA1#29,,SP-050518,withdrawn,SA#29,S1,,VCC
1799,C,Rel-7,Charging requirements for voice call continuity,S1-050897,agreed,SA1#29,,SP-050518,approved,SA#29,S1,,VCC


Now, we want to first link tdoc link and source involved, first crawl fast t-doc

In [ ]:
#!/usr/bin/env python3
"""
FAST bulk metadata for 3GPP TDocs — avoids downloading individual documents.

Instead of fetching ~500k TDoc files, this crawls the public 3GPP FTP tree and
reads, per meeting, just two small things:
  * the Docs/ directory listing  -> TDoc# + UPLOAD date (e.g. 2010/04/14 14:35)
  * the Docs/DocList.csv          -> TDoc# + Source

That is a few thousand small requests in total (one pair per meeting) rather
than ~1M, so it finishes in minutes instead of tens of hours.

NOTE on the date: this is the file's UPLOAD date, not the author-entered "Date"
inside the CR form (those differ by days). Use scrape_tdoc_source_date.py if you
need the exact in-document date.

Two steps:
  1. crawl  -> builds tdoc_meta.csv  (columns: tdoc, upload_date, upload_iso, source)
  2. join   -> merges that onto your clean2 CSV by WG TDoc # (fallback TSG TDoc #)

Resumable: visited directories and harvested rows are written incrementally;
re-running continues where it stopped.

Examples:
    # validate on one working group first (fast):
    python crawl_3gpp_meta.py crawl --root https://www.3gpp.org/ftp/TSG_SA/WG3_Security/
    # then the whole tree:
    python crawl_3gpp_meta.py crawl
    # finally attach to your data:
    python crawl_3gpp_meta.py join --input 3gpp_relcrs_clean2.csv \
        --meta tdoc_meta.csv --output clean2_with_meta.csv

Requirements: pip install pandas requests
"""
from __future__ import annotations

import argparse
import csv
import io
import os
import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from requests.adapters import HTTPAdapter

try:
    from urllib3.util.retry import Retry
except Exception:  # pragma: no cover
    Retry = None

FTP_ROOT = "https://www.3gpp.org/ftp/"
DEFAULT_ROOTS = [
    "https://www.3gpp.org/ftp/tsg_sa/", "https://www.3gpp.org/ftp/tsg_ct/",
    "https://www.3gpp.org/ftp/tsg_ran/", "https://www.3gpp.org/ftp/tsg_geran/",
    "https://www.3gpp.org/ftp/TSG_SA/", "https://www.3gpp.org/ftp/TSG_CT/",
    "https://www.3gpp.org/ftp/TSG_RAN/", "https://www.3gpp.org/ftp/TSG_GERAN/",
    "https://www.3gpp.org/ftp/TSG_T/", "https://www.3gpp.org/ftp/TSG_CN/",
]
HEADERS = {"User-Agent": "Mozilla/5.0 (3gpp-meta-crawler)"}
DATE_RE = re.compile(r"(\d{4})/(\d{1,2})/(\d{1,2})\s+(\d{1,2}):(\d{2})")
TDOC_RE = re.compile(r"^([A-Za-z]+\d*-\d+)")
FILE_EXT = (".zip", ".doc", ".docx", ".pdf", ".ppt", ".pptx")


# --------------------------------------------------------------------------- #
# Parsing
# --------------------------------------------------------------------------- #
def parse_listing(html: str, base_url: str) -> tuple[list[str], dict[str, str]]:
    """From a directory-listing page return (subdir_urls, {tdoc: upload_date}).
    Each table row holds an <a href> plus a date cell 'YYYY/MM/DD HH:MM'."""
    subdirs: list[str] = []
    files: dict[str, str] = {}
    base = base_url if base_url.endswith("/") else base_url + "/"
    for block in re.split(r"<tr[ >]", html)[1:]:
        m = re.search(r'href="([^"]+)"', block)
        if not m:
            continue
        href = m.group(1)
        if "sortby" in href or href.startswith("?"):
            continue
        url = href if href.startswith("http") else (
            "https://www.3gpp.org" + href if href.startswith("/")
            else base + href)
        name = url.rstrip("/").rsplit("/", 1)[-1]
        low = name.lower()
        if low.endswith(FILE_EXT):
            tid = TDOC_RE.match(name)
            d = DATE_RE.search(block)
            if tid and d:
                iso = (f"{int(d.group(1)):04d}-{int(d.group(2)):02d}-"
                       f"{int(d.group(3)):02d}")
                files[tid.group(1)] = iso          # one row per tdoc id
        elif "." not in name and url.rstrip("/") != base.rstrip("/"):
            subdirs.append(url.rstrip("/") + "/")  # a subfolder
    return subdirs, files


def parse_doclist(text: str) -> dict[str, str]:
    """{tdoc: source} from a DocList.csv (semicolon-delimited, quoted, with
    embedded newlines in some fields)."""
    out: dict[str, str] = {}
    rdr = csv.reader(io.StringIO(text), delimiter=";")
    try:
        header = next(rdr)
    except StopIteration:
        return out
    try:
        i_name = header.index("Doc. Name")
        i_src = header.index("Source")
    except ValueError:
        return out
    for row in rdr:
        if len(row) > max(i_name, i_src) and row[i_name].strip():
            tid = TDOC_RE.match(row[i_name].strip())
            if tid:
                out[tid.group(1)] = row[i_src].strip()
    return out


# --------------------------------------------------------------------------- #
# Networking
# --------------------------------------------------------------------------- #
def make_session(pool: int) -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    retry = None
    if Retry is not None:
        retry = Retry(total=4, backoff_factor=1.0,
                      status_forcelist=(429, 500, 502, 503, 504),
                      allowed_methods=frozenset({"GET"}))
    ad = HTTPAdapter(pool_connections=pool, pool_maxsize=pool, max_retries=retry)
    s.mount("https://", ad)
    return s


def get_text(url: str, session: requests.Session) -> str:
    r = session.get(url, timeout=(15, 120))
    r.raise_for_status()
    b = r.content
    if b[:2] in (b"\xff\xfe", b"\xfe\xff"):       # UTF-16 (some DocList.csv)
        return b.decode("utf-16", "ignore")
    if b[:3] == b"\xef\xbb\xbf":                    # UTF-8 BOM
        return b[3:].decode("utf-8", "ignore")
    return b.decode("utf-8", "ignore")


# --------------------------------------------------------------------------- #
# Crawl
# --------------------------------------------------------------------------- #
class MetaWriter:
    COLS = ["tdoc", "upload_iso", "source", "meeting_url"]

    def __init__(self, path: str):
        self.lock = threading.Lock()
        new = not os.path.exists(path)
        self.fh = open(path, "a", encoding="utf-8", newline="")
        self.w = csv.writer(self.fh)
        if new:
            self.w.writerow(self.COLS)

    def write_rows(self, rows):
        with self.lock:
            self.w.writerows(rows)
            self.fh.flush()

    def close(self):
        self.fh.close()


def _load_visited(path: str) -> set[str]:
    if not os.path.exists(path):
        return set()
    with open(path, encoding="utf-8") as fh:
        return {ln.strip() for ln in fh if ln.strip()}


def crawl(roots=None, workers=16, meta_path="tdoc_meta.csv",
          visited_path="crawl_visited.txt"):
    roots = roots or DEFAULT_ROOTS
    session = make_session(workers)
    visited = _load_visited(visited_path)
    seen_tdoc = set()
    if os.path.exists(meta_path):
        try:
            seen_tdoc = set(pd.read_csv(meta_path, usecols=["tdoc"],
                                        dtype=str)["tdoc"])
        except Exception:
            pass
    print(f"Resuming: {len(visited)} dirs visited, {len(seen_tdoc)} tdocs known")

    writer = MetaWriter(meta_path)
    vfh = open(visited_path, "a", encoding="utf-8")
    vlock = threading.Lock()
    frontier = [r for r in roots if r not in visited]
    stats = {"dirs": 0, "rows": 0}

    def visit(url):
        """Return list of newly discovered subdir urls; harvest tdoc rows."""
        try:
            html = get_text(url, session)
        except Exception:
            return []
        subdirs, files = parse_listing(html, url)
        rows = []
        if url.rstrip("/").endswith("/Docs"):
            # source map from DocList.csv if present in this Docs folder
            try:
                srcmap = parse_doclist(
                    get_text(url.rstrip("/") + "/DocList.csv", session))
            except Exception:
                srcmap = {}
            for tid, iso in files.items():
                if tid in seen_tdoc:
                    continue
                seen_tdoc.add(tid)
                rows.append([tid, iso, srcmap.get(tid, ""), url])
        if rows:
            writer.write_rows(rows)
            stats["rows"] += len(rows)
        return subdirs

    try:
        while frontier:
            nxt = []
            with ThreadPoolExecutor(max_workers=workers) as ex:
                futs = {ex.submit(visit, u): u for u in frontier
                        if u not in visited}
                for fut in as_completed(futs):
                    u = futs[fut]
                    visited.add(u)
                    with vlock:
                        vfh.write(u + "\n"); vfh.flush()
                    stats["dirs"] += 1
                    for sd in fut.result():
                        if sd not in visited:
                            nxt.append(sd)
                    if stats["dirs"] % 100 == 0:
                        print(f"  dirs={stats['dirs']} tdocs={stats['rows']} "
                              f"frontier={len(nxt)}", flush=True)
            frontier = list(dict.fromkeys(nxt))   # dedupe, keep order
    except KeyboardInterrupt:
        print("\nInterrupted; progress saved, re-run to resume.")
    finally:
        writer.close(); vfh.close()
    print(f"Done: {stats['dirs']} dirs, {stats['rows']} new tdoc rows -> {meta_path}")


# --------------------------------------------------------------------------- #
# Join
# --------------------------------------------------------------------------- #
def _cell(row, col):
    v = row.get(col)
    return str(v).strip() if (pd.notna(v) and str(v).strip()) else None


def join(input_csv, meta_csv, output_csv):
    df = pd.read_csv(input_csv)
    meta = pd.read_csv(meta_csv, dtype=str).drop_duplicates("tdoc", keep="last")
    m = meta.set_index("tdoc")
    up = m["upload_iso"].to_dict()
    sr = m["source"].to_dict() if "source" in m else {}

    def pick(row):
        for c in ("WG TDoc #", "TSG TDoc #"):
            u = _cell(row, c)
            if u and u in up:
                return u
        return _cell(row, "WG TDoc #") or _cell(row, "TSG TDoc #")

    uid = df.apply(pick, axis=1)
    df["tdoc_uid_used"] = uid
    df["upload_date"] = uid.map(up)
    df["doclist_source"] = uid.map(lambda u: sr.get(u) or None)

    # Unified source: existing CSV columns first (they are ~98.5% filled), then
    # the DocList source as a fallback for the gaps.
    def best_source(row):
        return (_cell(row, "WG Source information")
                or _cell(row, "TSG Source information")
                or (row["doclist_source"] if pd.notna(row["doclist_source"]) else None))
    df["source"] = df.apply(best_source, axis=1)

    df.to_csv(output_csv, index=False)
    hit = df["upload_date"].notna().sum()
    src = df["source"].notna().sum()
    print(f"Joined -> {output_csv}: upload_date {hit}/{len(df)}, "
          f"source {src}/{len(df)}")


def main():
    ap = argparse.ArgumentParser(description="Bulk 3GPP TDoc metadata via FTP")
    sub = ap.add_subparsers(dest="cmd")

    c = sub.add_parser("crawl")
    c.add_argument("--root", action="append",
                   help="start URL(s); default = all TSG roots")
    c.add_argument("--workers", type=int, default=16)
    c.add_argument("--meta", default="tdoc_meta.csv")
    c.add_argument("--visited", default="crawl_visited.txt")

    j = sub.add_parser("join")
    j.add_argument("--input", default="3gpp_relcrs_clean2.csv")
    j.add_argument("--meta", default="tdoc_meta.csv")
    j.add_argument("--output", default="clean2_with_meta.csv")

    try:
        args, _ = ap.parse_known_args()
    except SystemExit:
        # Happens when run inside a notebook (kernel args look like a bad
        # subcommand). Don't crash; functions can be called directly.
        print("Call crawl(...) or join(...) directly in a cell, or run from the "
              "command line with the 'crawl' / 'join' subcommand.")
        return
    if args.cmd == "crawl":
        crawl(args.root or DEFAULT_ROOTS, args.workers, args.meta, args.visited)
    elif args.cmd == "join":
        join(args.input, args.meta, args.output)
    else:
        # No/invalid subcommand (e.g. run inside a notebook). Don't crash;
        # the crawl()/join() functions are available to call directly.
        print("Call crawl(...) or join(...) directly, or use the 'crawl'/'join' "
              "subcommands from the command line.")


if __name__ == "__main__":
    main()

In [ ]:
crawl(None, 16, "tdoc_meta.csv", "crawl_visited.txt")

In [ ]:
#!/usr/bin/env python3
"""
Scrape `Source` and `Date` from 3GPP TDoc Change-Request files and add them to
the clean2 dataframe as columns: source_to_wg, source_to_tsg, date, date_iso
(normalised YYYY-MM-DD), tdoc_uid_used, scrape_status.

For each row the contribution id is taken from `WG TDoc #`; if the WG document
yields no usable source it falls back to `TSG TDoc #`. The id is appended to
    https://portal.3gpp.org/ngppapp/DownloadTDoc.aspx?contributionUid=<id>
which returns an HTML landing page that JS-redirects to the real file on the
3GPP FTP server (a .zip, sometimes zip-wrapped, or a direct .doc/.docx). The
CR-form file is parsed for the source(s) and date.

Built for the full ~500k-row dataset:
  * DEDUPED  - each unique contribution id is downloaded only once.
  * RESUMABLE- every result is appended to a cache CSV (default tdoc_cache.csv);
               re-running skips ids already in the cache, so the job can be
               stopped/resumed or run in batches and never repeats work.
  * CONCURRENT - many downloads in flight via a tuned, connection-pooled,
               auto-retrying requests.Session.
  * IN-MEMORY - files are parsed from RAM (no temp files), avoiding disk churn.
  * CSV FALLBACK - if a document gives no source, the existing
               'WG/TSG Source information' columns fill it in.

Requirements:
    pip install pandas requests
    pip install python-docx   # optional; cleaner .docx text (stdlib fallback ok)

Examples:
    python scrape_tdoc_source_date.py --limit 0 --workers 48
    python scrape_tdoc_source_date.py --offset 100000 --limit 100000   # a batch
"""
from __future__ import annotations

import argparse
import io
import os
import re
import threading
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from requests.adapters import HTTPAdapter

try:
    from urllib3.util.retry import Retry
except Exception:  # pragma: no cover
    Retry = None

BASE_URL = "https://portal.3gpp.org/ngppapp/DownloadTDoc.aspx?contributionUid="
DOC_EXTS = (".doc", ".docx")
HEADERS = {"User-Agent": "Mozilla/5.0 (TDoc-scraper)"}
REDIRECT_RE = re.compile(rb"window\.location\.href\s*=\s*'([^']+)'")

# Numeric codes for the scrape_status column (see legend printed by run()).
STATUS_CODES = {
    "ok": 1,                       # source found in the document
    "cr_but_no_source_field": 2,   # CR form found, but no source filled in it
    "not_a_change_request": 3,     # a file was found, but it is not a CR form
    "no_file_on_portal": 4,        # no document available on the portal/FTP
    "download_error": 5,           # download failed after retries
    "no_uid": 0,                   # row had no WG/TSG TDoc # at all
}

_BOUND = (r"(?:Source to WG\s*:|Source to TSG\s*:|Work item code\s*:|Date\s*:|"
          r"Category\s*:|Release\s*:|Title\s*:|Reason for change\s*:|"
          r"Summary of change\s*:)")

# Date can be numeric (23/02/2005, 2008-11-27) or have a textual month
# (11 May, 2001 / May 11, 2001 / 3rd Mar 2010 / 1 September 1999).
_MONTH = (r"Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|"
          r"Jul(?:y)?|Aug(?:ust)?|Sep(?:t)?(?:ember)?|Oct(?:ober)?|"
          r"Nov(?:ember)?|Dec(?:ember)?")
_DATE = (r"\d{4}[-/.]\d{1,2}[-/.]\d{1,2}"               # ISO 2008-11-27
         r"|\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4}"            # 23/02/2005, 11-22-00
         r"|[0-3]?\d(?:st|nd|rd|th)?\s+(?:" + _MONTH + r")\.?,?\s+\d{2,4}"
         r"|(?:" + _MONTH + r")\.?\s+[0-3]?\d(?:st|nd|rd|th)?,?\s+\d{2,4}")


# --------------------------------------------------------------------------- #
# Parsing
# --------------------------------------------------------------------------- #
def _normalize(text: str) -> str:
    chars = (" " if (ord(c) < 0x20 or 0xF000 <= ord(c) <= 0xF0FF) else c
             for c in text)
    return re.sub(r"[ \t]+", " ", "".join(chars))


def _parse_cover(norm: str) -> tuple[str | None, str | None, str | None]:
    """Fallback for non-CR documents that are TSG cover sheets / CR-packs, e.g.
        '3GPP TSG-SA Meeting #38  SP-070913  3-6 December 2007, Cancun, Mexico
         Source: TSG SA  Title: CRs on 22.011 ...  Agenda item: 9.41'
    These have no 'CHANGE REQUEST' table and no 'Date:' label; the date is the
    meeting date in the header. Returns the cover Source as source_to_wg and the
    meeting date (its end day for a range, e.g. '6 December 2007')."""
    start = norm.find("3GPP TSG")
    region = norm[start:start + 700] if start != -1 else norm[:700]
    if "source" not in region.lower():
        return None, None, None
    src = None
    m = re.search(r"Source\s*:\s*(.+?)\s*"
                  r"(?:Title\s*:|Agenda|Document for|WG\s*:|$)", region, re.I)
    if m:
        src = re.sub(r"\s+", " ", m.group(1)).strip(" :-") or None
    dm = re.search(r"(" + _DATE + r")", region, re.I)
    date = dm.group(1).strip() if dm else None
    return (src, None, date) if (src or date) else (None, None, None)


def parse_source_date(text: str) -> tuple[str | None, str | None, str | None]:
    """Return (source_to_wg, source_to_tsg, date). Handles both CR-form
    generations (single 'Source:' -> WG; or 'Source to WG/TSG:'), and TSG cover
    sheets via _parse_cover when there is no CHANGE REQUEST table."""
    norm = _normalize(text)
    pos = norm.upper().find("CHANGE REQUEST")
    if pos == -1:
        return _parse_cover(norm)   # e.g. a TSG cover sheet / CR-pack
    norm = norm[pos:]  # search inside the CR table, not any cover header

    def grab(label: str) -> str | None:
        m = re.search(label + r"\s*[\(\)\s]*([^\n]*?)\s*" + _BOUND, norm, re.I)
        return (m.group(1).strip(" ()") or None) if m else None

    src_wg = grab(r"Source to WG\s*:")
    src_tsg = grab(r"Source to TSG\s*:")
    if not src_wg and not src_tsg:
        src_wg = grab(r"\bSource\s*:")

    date = None
    m = re.search(r"Date\s*:\s*[\(\)\s]*(" + _DATE + r")", norm, re.I)
    if m:
        date = m.group(1).strip()
    return src_wg, src_tsg, date


_MONTH_NUM = {abbr: i for i, abbr in enumerate(
    ["jan", "feb", "mar", "apr", "may", "jun",
     "jul", "aug", "sep", "oct", "nov", "dec"], 1)}


def _yy(year: str) -> int:
    y = int(year)
    if y >= 100:
        return y
    return 1900 + y if y >= 90 else 2000 + y   # 99->1999, 00..26->2000s


def to_iso(raw: str | None) -> str | None:
    """Normalise a raw CR date string to YYYY-MM-DD. For ambiguous numeric
    dates where neither part exceeds 12 (e.g. 03-04-2005) the European
    day-first reading is assumed, matching the bulk of 3GPP submissions."""
    if not raw:
        return None
    s = str(raw).strip()
    m = re.match(r"(\d{1,2})(?:st|nd|rd|th)?\s+([A-Za-z]+)\.?,?\s+(\d{2,4})$", s)
    if m:
        d, mon, y = m.groups()
        mm = _MONTH_NUM.get(mon[:3].lower())
        if mm:
            return f"{_yy(y):04d}-{mm:02d}-{int(d):02d}"
    m = re.match(r"([A-Za-z]+)\.?\s+(\d{1,2})(?:st|nd|rd|th)?,?\s+(\d{2,4})$", s)
    if m:
        mon, d, y = m.groups()
        mm = _MONTH_NUM.get(mon[:3].lower())
        if mm:
            return f"{_yy(y):04d}-{mm:02d}-{int(d):02d}"
    m = re.match(r"(\d{4})[-/.](\d{1,2})[-/.](\d{1,2})$", s)
    if m:
        y, a, b = m.groups()
        return f"{int(y):04d}-{int(a):02d}-{int(b):02d}"
    m = re.match(r"(\d{1,2})[-/.](\d{1,2})[-/.](\d{2,4})$", s)
    if m:
        a, b, y = int(m.group(1)), int(m.group(2)), m.group(3)
        if a > 12 and b <= 12:        # DD/MM
            d, mo = a, b
        elif b > 12 and a <= 12:      # MM/DD
            mo, d = a, b
        else:                          # ambiguous -> day-first
            d, mo = a, b
        if 1 <= mo <= 12 and 1 <= d <= 31:
            return f"{_yy(y):04d}-{mo:02d}-{d:02d}"
    return None


# --------------------------------------------------------------------------- #
# Text extraction (in-memory)
# --------------------------------------------------------------------------- #
def _docx_bytes_to_text(data: bytes) -> str:
    try:
        from docx import Document
        doc = Document(io.BytesIO(data))
        parts = [p.text for p in doc.paragraphs]
        for tbl in doc.tables:
            for row in tbl.rows:
                cells, last = [], None
                for c in row.cells:           # collapse merged (repeated) cells
                    if c._tc is last:
                        continue
                    last = c._tc
                    cells.append(c.text)
                parts.append("\t".join(cells))
        return "\n".join(parts)
    except Exception:
        import html
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            xml = z.read("word/document.xml").decode("utf-8", "ignore")
        xml = re.sub(r"<w:instrText[^>]*>.*?</w:instrText>", "", xml, flags=re.S)
        xml = xml.replace("</w:p>", "\n").replace("</w:tc>", "\t")
        txt = html.unescape(re.sub(r"<[^>]+>", "", xml))
        txt = re.sub(r"DOCPROPERTY\s+\S+\s*", "", txt)
        return re.sub(r"\\\*\s*MERGEFORMAT", "", txt)


def _doc_bytes_to_text(data: bytes) -> str:
    """Legacy .doc text. The body is stored either as cp1252 (most CR forms) or
    UTF-16LE (e.g. TSG cover sheets). Decode both and keep whichever yields the
    most recognisable form/cover labels."""
    cands = [data.decode("cp1252", "ignore"),
             data.decode("utf-16-le", "ignore")]

    def score(t: str) -> int:
        u = t.upper()
        return sum(u.count(k) for k in (
            "CHANGE REQUEST", "SOURCE:", "TITLE:", "DATE:", "AGENDA",
            "3GPP", "WORK ITEM", "MEETING"))
    return max(cands, key=score)


def _office_text(kind: str, blob: bytes) -> str:
    return _docx_bytes_to_text(blob) if kind == "docx" else _doc_bytes_to_text(blob)


def _iter_office(data: bytes, depth: int = 0):
    """Yield (kind, bytes) for every real Office doc inside `data`, recursing
    through zip wrappers. Inner filename need not match the id; there may be
    several files."""
    if depth > 6:
        return
    if data[:4] == b"\xd0\xcf\x11\xe0":
        yield "doc", data
        return
    if data[:2] == b"PK":
        try:
            zf = zipfile.ZipFile(io.BytesIO(data))
        except zipfile.BadZipFile:
            return
        names = zf.namelist()
        if "word/document.xml" in names or "[Content_Types].xml" in names:
            yield "docx", data
            return
        for n in names:
            if n.startswith("__MACOSX") or n.endswith("/"):
                continue
            if n.lower().endswith(DOC_EXTS):
                yield from _iter_office(zf.read(n), depth + 1)


# --------------------------------------------------------------------------- #
# Networking
# --------------------------------------------------------------------------- #
def make_session(pool: int) -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    retry = None
    if Retry is not None:
        retry = Retry(total=4, backoff_factor=1.0,          # 1s,2s,4s,8s backoff
                      status_forcelist=(429, 500, 502, 503, 504),
                      allowed_methods=frozenset({"GET"}),
                      respect_retry_after_header=True)
    adapter = HTTPAdapter(pool_connections=pool, pool_maxsize=pool,
                          max_retries=retry)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s


def _looks_office(b: bytes) -> bool:
    return b[:2] == b"PK" or b[:4] == b"\xd0\xcf\x11\xe0"


def fetch_uid(uid: str, session: requests.Session,
              meeting_url: str | None = None) -> dict:
    """Download + parse one contribution entirely in memory.
    Returns {wg, tsg, date, status}.

    If meeting_url is known (from the FTP crawl, tdoc_meta.csv), the file is
    fetched DIRECTLY as <meeting_url>/<uid>.zip — skipping the portal landing
    page, i.e. one request instead of two. Falls back to the portal flow when
    the direct URL is missing or doesn't return a real document."""
    out = {"wg": None, "tsg": None, "date": None, "status": "no_file_on_portal"}
    try:
        content = None
        if meeting_url:
            direct = meeting_url.rstrip("/") + "/" + uid + ".zip"
            r = session.get(direct, timeout=(15, 240))
            if r.status_code == 200 and _looks_office(r.content):
                content = r.content

        if content is None:  # portal landing -> JS redirect -> real file
            r = session.get(BASE_URL + uid, timeout=(15, 90))
            r.raise_for_status()
            content = r.content
            if b"<html" in content[:600].lower():
                m = REDIRECT_RE.search(content)
                if not m:
                    return out
                real_url = m.group(1).decode("utf-8", "ignore").strip()
                r = session.get(real_url, timeout=(15, 240))
                r.raise_for_status()
                content = r.content

        docs = list(_iter_office(content))
        if not docs:
            return out

        best, is_cr = (None, None, None), False
        for kind, blob in docs:
            txt = _office_text(kind, blob)
            if "CHANGE REQUEST" in _normalize(txt).upper():
                is_cr = True
            w, t, d = parse_source_date(txt)
            if w or t or d:
                best = (w, t, d)
            if (w or t) and d:
                break
        out["wg"], out["tsg"], out["date"] = best
        if best[0] or best[1]:
            out["status"] = "ok"
        elif not is_cr:
            out["status"] = "not_a_change_request"
        else:
            out["status"] = "cr_but_no_source_field"
    except Exception as e:  # noqa: BLE001
        out["status"] = "download_error"
        out["error"] = str(e)[:200]
    return out


# --------------------------------------------------------------------------- #
# Cache (resumable)
# --------------------------------------------------------------------------- #
_CACHE_COLS = ["uid", "source_to_wg", "source_to_tsg", "date", "scrape_status"]


def load_cache(path: str) -> dict[str, dict]:
    if not path or not os.path.exists(path):
        return {}
    try:
        c = pd.read_csv(path, dtype=str, keep_default_na=False)
    except Exception:
        return {}
    cache = {}
    for _, r in c.iterrows():
        cache[r["uid"]] = {
            "wg": r.get("source_to_wg") or None,
            "tsg": r.get("source_to_tsg") or None,
            "date": r.get("date") or None,
            "status": r.get("scrape_status") or None,
        }
    return cache


class CacheWriter:
    """Thread-safe incremental append to the cache CSV."""
    def __init__(self, path: str):
        self.path = path
        self.lock = threading.Lock()
        self.fh = None
        if path:
            new = not os.path.exists(path)
            self.fh = open(path, "a", encoding="utf-8", newline="")
            if new:
                self.fh.write(",".join(_CACHE_COLS) + "\n")

    def write(self, uid: str, res: dict):
        if not self.fh:
            return
        def esc(v):
            v = "" if v is None else str(v)
            return '"' + v.replace('"', '""') + '"' if any(ch in v for ch in ',"\n') else v
        row = [uid, res.get("wg"), res.get("tsg"), res.get("date"), res.get("status")]
        line = ",".join(esc(v) for v in row) + "\n"
        with self.lock:
            self.fh.write(line)
            self.fh.flush()

    def close(self):
        if self.fh:
            self.fh.close()


# --------------------------------------------------------------------------- #
# Orchestration
# --------------------------------------------------------------------------- #
def _cell(row: pd.Series, col: str) -> str | None:
    val = row.get(col)
    return str(val).strip() if (pd.notna(val) and str(val).strip()) else None


def _load_meta(meta_csv: str | None) -> tuple[dict[str, str], dict[str, str]]:
    """From a crawl tdoc_meta.csv return ({tdoc: meeting_url}, {tdoc: upload_iso}).
    meeting_url lets us fetch files directly (skip the portal); upload_iso is the
    crawl's upload date, used as a fallback when a document yields no date."""
    if not meta_csv or not os.path.exists(meta_csv):
        return {}, {}
    try:
        m = pd.read_csv(meta_csv, dtype=str)
    except Exception:
        return {}, {}
    urls = dict(zip(m["tdoc"], m["meeting_url"])) if "meeting_url" in m else {}
    ups = dict(zip(m["tdoc"], m["upload_iso"])) if "upload_iso" in m else {}
    return urls, ups


def run(input_csv: str, output_csv: str, limit: int | None, workers: int,
        offset: int = 0, cache_path: str = "tdoc_cache.csv",
        meta_csv: str | None = "tdoc_meta.csv") -> None:
    df = pd.read_csv(input_csv)
    end = None if not limit else offset + limit
    work = df.iloc[offset:end]
    print(f"Rows in scope: {len(work)} (offset={offset}, limit={limit or 'all'})")

    wg_uid = {i: _cell(r, "WG TDoc #") for i, r in work.iterrows()}
    tsg_uid = {i: _cell(r, "TSG TDoc #") for i, r in work.iterrows()}
    primary = {i: (wg_uid[i] or tsg_uid[i]) for i in work.index}

    meeting_urls, upload_dates = _load_meta(meta_csv)
    if meeting_urls:
        print(f"Using {len(meeting_urls)} direct file URLs from {meta_csv} "
              f"(skips the portal landing page).")

    cache = load_cache(cache_path)
    writer = CacheWriter(cache_path)
    session = make_session(workers)

    def has_source(uid: str | None) -> bool:
        r = cache.get(uid)
        return bool(r and (r.get("wg") or r.get("tsg")))

    def fetch_ids(ids: set[str], label: str) -> None:
        # A cached 'download_error' is retried; anything else cached is skipped.
        todo = [u for u in ids
                if u and (u not in cache
                          or cache[u].get("status") == "download_error")]
        retried = sum(1 for u in todo if u in cache)
        print(f"{label}: {len(ids)} unique ids | to download {len(todo)} "
              f"(retried errors: {retried})", flush=True)
        done = 0
        with ThreadPoolExecutor(max_workers=workers) as ex:
            futs = {ex.submit(fetch_uid, u, session, meeting_urls.get(u)): u
                    for u in todo}
            for fut in as_completed(futs):
                u = futs[fut]
                cache[u] = fut.result()
                writer.write(u, cache[u])
                done += 1
                if done % 200 == 0 or done == len(todo):
                    print(f"  downloaded {done}/{len(todo)}", flush=True)

    try:
        # Phase 1: the primary id of every row (WG, or TSG when WG is empty).
        fetch_ids({primary[i] for i in work.index if primary[i]}, "Phase 1 (WG)")
        # Phase 2 (extra-safe fallback): only rows whose primary gave NO source
        # AND that have a *different* TSG id — so we don't download TSG needlessly.
        fallback = {tsg_uid[i] for i in work.index
                    if tsg_uid[i] and tsg_uid[i] != primary[i]
                    and not has_source(primary[i])}
        fetch_ids(fallback, "Phase 2 (TSG fallback)")
    except KeyboardInterrupt:
        print(f"\nInterrupted. Progress saved to {cache_path}; re-run to resume.")
    finally:
        writer.close()

    # Resolve each row: prefer the id that yielded a source (WG first, then TSG).
    def resolve(i):
        for uid in (primary[i], tsg_uid[i]):
            if uid and uid in cache:
                r = cache[uid]
                if r.get("wg") or r.get("tsg"):
                    return uid, r
        # none had a source: return the primary record for status/date
        uid = primary[i]
        return uid, cache.get(uid, {"wg": None, "tsg": None, "date": None,
                                    "status": "no_uid" if not uid else None})

    out = work.copy()
    res_by_row = {i: resolve(i) for i in work.index}
    out["source_to_wg"] = out.index.map(lambda i: res_by_row[i][1].get("wg"))
    out["source_to_tsg"] = out.index.map(lambda i: res_by_row[i][1].get("tsg"))
    out["date"] = out.index.map(lambda i: res_by_row[i][1].get("date"))
    out["tdoc_uid_used"] = out.index.map(lambda i: res_by_row[i][0])
    out["scrape_status"] = out.index.map(lambda i: res_by_row[i][1].get("status"))

    # Date provenance: from the document, else fall back to the crawl's upload
    # date (tdoc_meta.csv) so rows like 'cr_but_no_source_field' / no-date still
    # get a date.
    out["date_origin"] = out["date"].map(lambda d: "document" if pd.notna(d)
                                         and str(d).strip() else None)
    if upload_dates:
        need = out["date_origin"].isna()
        out.loc[need, "date"] = out.loc[need, "tdoc_uid_used"].map(
            lambda u: upload_dates.get(u))
        newly = need & out["date"].notna() & out["date_origin"].isna()
        out.loc[newly, "date_origin"] = "upload"
    out["date_iso"] = out["date"].map(to_iso)

    # Numeric encodings of the status fields (legend in STATUS_CODES). Keep only
    # the numeric columns; drop the string versions.
    out["status_code"] = out["scrape_status"].map(STATUS_CODES).astype("Int64")
    out["date_is_exact"] = (out["date_origin"].map({"document": 1, "upload": 0})
                            .astype("Int64"))
    status_counts = dict(out["status_code"].value_counts(dropna=False))
    out = out.drop(columns=["scrape_status", "date_origin"])

    # Fallback: fill missing source from the existing CSV columns.
    def fill(row, target, col):
        if row[target]:
            return row[target]
        return _cell(row, col)
    if "WG Source information" in out.columns:
        out["source_to_wg"] = out.apply(
            lambda r: fill(r, "source_to_wg", "WG Source information"), axis=1)
    if "TSG Source information" in out.columns:
        out["source_to_tsg"] = out.apply(
            lambda r: fill(r, "source_to_tsg", "TSG Source information"), axis=1)

    out.to_csv(output_csv, index=False)
    ok = int((out["source_to_wg"].notna() | out["source_to_tsg"].notna()).sum())
    print(f"\nWrote {output_csv}")
    print(f"  rows with a source (incl. CSV fallback): {ok}/{len(out)}")
    print("  status_code counts:", status_counts)
    print("  status_code legend:", {v: k for k, v in STATUS_CODES.items()})
    print("  date_is_exact: 1 = exact date from document, "
          "0 = upload-date fallback, blank = no date")


def main() -> None:
    ap = argparse.ArgumentParser(description="Scrape Source/Date from 3GPP TDocs")
    ap.add_argument("--input", default="3gpp_relcrs_clean2.csv")
    ap.add_argument("--output", default="clean2_with_source_date.csv")
    ap.add_argument("--limit", type=int, default=10,
                    help="rows to process (0 = all)")
    ap.add_argument("--offset", type=int, default=0,
                    help="start row (for batching)")
    ap.add_argument("--workers", type=int, default=32)
    ap.add_argument("--cache", default="tdoc_cache.csv",
                    help="resumable cache file of per-id results")
    ap.add_argument("--meta", default="tdoc_meta.csv",
                    help="crawl tdoc_meta.csv -> direct file URLs (2x faster). "
                         "Set to '' to always use the portal.")
    args, _ = ap.parse_known_args()
    run(args.input, args.output, args.limit or None, args.workers,
        offset=args.offset, cache_path=args.cache, meta_csv=args.meta or None)


if __name__ == "__main__":
    main()

In [ ]:
#276805
run(
    "3gpp_relcrs.csv", "3ggp_with_rel18_linked_tdoc_and_source.csv",
    limit=276805, workers=45, cache_path="tdoc_cache.csv"
)

In [21]:
df = pd.read_csv("3gpp_rel_new_cleaned.csv")
clean = df[df["Target Release"] == "Rel-18"]
clean.to_csv("3gpp_relcrs_cleaned_rel18New.csv", index=False)

In [22]:
#276805
run(
    "3gpp_relcrs_cleaned_rel18New.csv", "clean3ggp_with_rel18_source_date_wo_revised.csv",
    limit=276805, workers=45, cache_path="tdoc_cache.csv"
)

Rows in scope: 68382 (offset=0, limit=276805)
Using 1476838 direct file URLs from tdoc_meta.csv (skips the portal landing page).
Phase 1 (WG): 68166 unique ids | to download 35732 (retried errors: 9)
  downloaded 200/35732
  downloaded 400/35732
  downloaded 600/35732
  downloaded 800/35732
  downloaded 1000/35732
  downloaded 1200/35732
  downloaded 1400/35732
  downloaded 1600/35732
  downloaded 1800/35732
11:01:42 | WARNING | Retrying (Retry(total=3, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ReadTimeoutError("HTTPSConnectionPool(host='portal.3gpp.org', port=443): Read timed out. (read timeout=90)")': /ngppapp/DownloadTDoc.aspx?contributionUid=S2-2302268
11:01:45 | WARNING | Retrying (Retry(total=3, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ReadTimeoutError("HTTPSConnectionPool(host='portal.3gpp.org', port=443): Read timed out. (read timeout=90)")': /ngppapp/DownloadTDoc.aspx?contributionUid=s3i230389
11: